In [31]:
from datasets import load_dataset

ds = load_dataset("gmanolache/CrypticBio", split="train")

Resolving data files:   0%|          | 0/631 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/180 [00:00<?, ?it/s]

In [32]:
print(ds)
ds[0]

Dataset({
    features: ['scientificName', 'year', 'month', 'day', 'decimalLatitude', 'decimalLongitude', 'url', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'vernacularName', 'crypticGroup'],
    num_rows: 171832985
})


{'scientificName': 'Salticus scenicus',
 'year': 2022.0,
 'month': 3.0,
 'day': 16.0,
 'decimalLatitude': 49.682217,
 'decimalLongitude': -124.975088,
 'url': 'https://inaturalist-open-data.s3.amazonaws.com/photos/183187792/original.jpg',
 'kingdom': 'Animalia',
 'phylum': 'Arthropoda',
 'class': 'Arachnida',
 'order': 'Araneae',
 'family': 'Salticidae',
 'genus': 'Salticus',
 'vernacularName': ['Zebra Jumper',
  'Zebra Spider',
  'Zebra Jumping Spider',
  'Zebra Back Spider'],
 'crypticGroup': ['Phidippus audax',
  'Salticus cingulatus',
  'Paraphidippus aurantius',
  'Platycryptus undatus',
  'Attulus fasciger',
  'Naphrys pulex',
  'Marpissa muscosa',
  'Eris militaris',
  'Menemerus semilimbatus',
  'Menemerus bivittatus',
  'Hasarius adansoni',
  'Phidippus putnami',
  'Pelegrina galathea',
  'Maevia inclemens',
  'Pseudeuophrys lanigera',
  'Phidippus otiosus']}

# We need to filter to a smaller sub-set of data so that experimetns are more managable 

In [33]:

#Load just the benchmark files
common =load_dataset("gmanolache/CrypticBio", data_files="CrypticBio-Benchmark/CrypticBio-Common.csv", split="train")
print(common.features)

{'scientificName': Value('string'), 'year': Value('float64'), 'month': Value('float64'), 'day': Value('float64'), 'decimalLatitude': Value('float64'), 'decimalLongitude': Value('float64'), 'url': Value('string'), 'kingdom': Value('string'), 'phylum': Value('string'), 'class': Value('string'), 'order': Value('string'), 'family': Value('string'), 'genus': Value('string'), 'vernacularName': Value('string')}


In [34]:
#get unique pairs from the benchmark
unique_names = set(common["scientificName"])

#batch filtering to get the common subset
common_subset = ds.filter(
    lambda x: [
        name in unique_names 
        for name in x['scientificName']
    ],
    batched=True,
    batch_size=10000
)


In [35]:
import requests
from PIL import Image
from io import BytesIO
import torch
import numpy as np

# Open the image from the URL
def open_image(url : list[str]):
    try:
        response = requests.get(url, timeout=10)
        return Image.open(BytesIO(response.content))
    except requests.exceptions.RequestException as e:
        print(f"Error downloading image: {e}")
        return None

# Example usage
img_url = common[12985]['url']
image = open_image(img_url)
image.show()

In [ ]:
import open_clip

# Load the model and preprocessors
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms('hf-hub:imageomics/bioclip')
tokenizer = open_clip.get_tokenizer('hf-hub:imageomics/bioclip')


#tojenize the text from family and genus - same as cryptic group names
def tokenize_family_genius(sample: dict):
    family_name = sample['family']
    genus_name = sample['genus']
    return tokenizer(family_name + ' ' + genus_name)

#preprocess the image and add batch dimension  
def preprocess_image(sample: dict):
    img_url = sample['url']
    image = open_image(img_url)
    if image is not None:
        return preprocess_val(image).unsqueeze(0)  #add batch dimension
    else:
        return None






# Test model on latin names


In [37]:
sample_index = np.random.randint(0, len(common_subset))
example = common_subset[sample_index]
list_of_confusing_species = example['crypticGroup'] + [f"{example['family']} {example['genus']}"]

tokenized_text = tokenizer(list_of_confusing_species)
image_features = preprocess_image(example)
with torch.no_grad(), torch.cuda.amp.autocast():
    image_features = model.encode_image(image_features)
    text_features = model.encode_text(tokenized_text)
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)
    text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)

print("Label probs:", text_probs) 
print(f"Ground truth: {example['family']} {example['genus']}")
print(f"Ground truth prob: {text_probs[0, -1].item():.4f}")
print(f"Predicted: {list_of_confusing_species[text_probs.argmax().item()]}")
print(f"Correct: {text_probs.argmax().item() == len(list_of_confusing_species) - 1}")

C:\Users\ivanb\AppData\Local\Temp\ipykernel_47484\2774734933.py:7: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():
c:\Users\ivanb\anaconda3\envs\rl_course\Lib\site-packages\torch\cuda\amp\autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


Label probs: tensor([[4.6147e-01, 1.0448e-05, 1.6506e-05, 1.2810e-03, 5.8597e-03, 4.4561e-01,
         1.9435e-10, 9.7727e-04, 5.2762e-06, 3.5814e-02, 5.3697e-10, 2.0323e-06,
         1.1030e-08, 2.8666e-12, 6.5409e-04, 2.7184e-06, 3.6668e-05, 4.7421e-02,
         4.6615e-07, 1.3383e-07, 1.1717e-09, 2.3304e-04, 6.0322e-04]])
Ground truth: Amanitaceae Amanita
Ground truth prob: 0.0006
Predicted: Amanita muscaria
Correct: False


# Translate the latin names of confusing species to the vernacularName and repeat check for similarity with a traget image

In [ ]:
#Took to long to run and crashed kernel, scrapped for now, not an interesting experiment anyway, but we can come back to it later 

## Experiment 1 - adding Date (month + day) to the existent embeddings 

Encoding the date(month,day), using sine and cosine following the intuition from: https://harrisonpim.com/blog/the-best-way-to-encode-dates-times-and-other-cyclical-features . We kinda mimic the first stage of the BioClip training - learing meaningful represemntation of date

In [39]:
def encode_date(month, day):
    """
    Encode month and day as cyclical features using sin/cos projections.
    Each cyclical component maps to a point on a unit circle,
    so December (12) is close to January (1), and day 365 is close to day 1.
    """
    # Normalize to [0, 1] range
    month_norm = (month - 1) / 12       
    day_of_year_norm = ((month - 1) * 30 + day) / 365 

    # Project onto unit circle using sin/cos
    month_sin = (np.sin(2 * np.pi * month_norm) + 1) / 2
    month_cos = (np.cos(2 * np.pi * month_norm) + 1) / 2
    day_sin = (np.sin(2 * np.pi * day_of_year_norm) + 1) / 2
    day_cos = (np.cos(2 * np.pi * day_of_year_norm) + 1) / 2

    return torch.tensor([month_sin, month_cos, day_sin, day_cos], dtype=torch.float32)

In [40]:
example = common_subset[0]
date_features = encode_date(example['month'], example['day'])
print(date_features)

tensor([0.5000, 0.0000, 0.3682, 0.0177])


Having designed our embeding space, we try to map the date to the same embeddiong space as the CLIP representations for image and text.

In [45]:
import torch.nn as nn

# This network is prett naive and prob can be improved

class DateEmbedder(nn.Module):
    """Maps 4-dim date features into BioCLIP's 512-dim space."""
    def __init__(self, date_dim=4, clip_dim=512):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(date_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, clip_dim),
        )

    def forward(self, x):
        emb = self.network(x)
        return emb / emb.norm(dim=-1, keepdim=True)  # L2 normalize to match CLIP space

**The idea for now is the following: Using the learned text represetnations from taxa, we can try to align the date as with it. Thus due to the rule of assosiativity, we can avoid the slow process of training with images! - although its probably a good idea to redsign the experiment**

In [ ]:
from tqdm import tqdm #to monitor progress

#use cude 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# returns the full name of the taxonomy for a sample
def taxonomy_string(sample):
    return f"{sample['kingdom']} {sample['phylum']} {sample['class']} {sample['order']} {sample['family']} {sample['genus']}"

# Collect unique taxonomy strings per species
species_to_taxonomy = {}
for sample in tqdm(common_subset, desc="Collecting taxonomy"):
    name = sample['scientificName']
    if name not in species_to_taxonomy:
        species_to_taxonomy[name] = taxonomy_string(sample)

# Sort species and corresponding taxonomy strings for consistent ordering
all_species = sorted(species_to_taxonomy.keys())
# Create a list of taxonomy strings in the same order as species
all_taxonomy_strings = [species_to_taxonomy[s] for s in all_species]

# Encode using BioCLI in batches
BATCH = 256
species_text_embs = []
with torch.no_grad(): 
    for i in tqdm(range(0, len(all_taxonomy_strings), BATCH), desc="Encoding taxonomy"):
        batch = all_taxonomy_strings[i:i+BATCH]
        tokens = tokenizer(batch).to(DEVICE)
        #normalize the embeddings to unit length to match CLIP space
        embs = model.encode_text(tokens)
        embs /= embs.norm(dim=-1, keepdim=True)
        species_text_embs.append(embs.cpu()) #move to CPU to save GPU memory

# Concatenate all batches into a single tensor
species_text_embs = torch.cat(species_text_embs, dim=0)
# Create a mapping from species name to index in the embedding tensor
label_to_idx = {s: i for i, s in enumerate(all_species)}
print(f"{len(all_species)} species, shape: {species_text_embs.shape}")

Encoding taxonomy: 100%|██████████| 1/1 [00:01<00:00,  1.81s/it]

158 species, shape: torch.Size([158, 512])


In [46]:
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split


 
# Precompute all date tensors and labels into memory for fast access during training
all_dates = [] 
all_labels = [] 
for i in tqdm(range(len(common_subset)), desc="Precomputing"): 
    row = common_subset[i] 
    if row['month'] is None or row['day'] is None: 
        continue 
    all_dates.append(encode_date(row['month'], row['day'])) 
    all_labels.append(label_to_idx[row['scientificName']]) 
#stack into tensors
all_dates = torch.stack(all_dates)          # (N, 4) 
all_labels = torch.tensor(all_labels)       # (N,) 
 
#mix the dates so that we have a nice split of the data, and to avoid any potential ordering bias during training
train_idx, test_idx = train_test_split(
    range(len(all_dates)),
    test_size=0.2,
    stratify=all_labels.numpy(),
    random_state=42
)
#Create train and test splits
train_dates, test_dates = all_dates[train_idx], all_dates[test_idx]
train_labels, test_labels = all_labels[train_idx], all_labels[test_idx]

# define a dataloader for the training set, maybe batch size is too big, I increased it to speed up training
train_loader = DataLoader(
    torch.utils.data.TensorDataset(train_dates, train_labels),
    batch_size=2048, shuffle=True
)

# Train  - TODO find a way to make run faster
date_embedder = DateEmbedder().to(DEVICE)
optimizer = torch.optim.Adam(date_embedder.parameters(), lr=1e-3)
species_text_embs_gpu = species_text_embs.to(DEVICE)

for epoch in range(30):
    total_loss = 0
    steps = 0
    for date_inputs, label_indices in train_loader:
        if len(date_inputs) < 2:
            continue
        date_inputs = date_inputs.to(DEVICE)
        #take the corresponding full taxa name embeddings for each label in the batch
        batch_text = species_text_embs_gpu[label_indices]
        date_embs = date_embedder(date_inputs)
        #get the cosine similarity between the date embeddings and the text embeddings 
      
        logits = date_embs @ batch_text.T
        labels = torch.arange(len(date_inputs)).to(DEVICE)
        #maybe its a flawed loss - Not sure of the CLIP similarity matrix makes sense with date since date (season) can be mapped to multuiple species, so we cannot just associate each date with a single species in the embedding space TODO - Think about that maybe a simpler approach exists!

        loss = nn.CrossEntropyLoss()(logits.T, labels)
        #perform optimization step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        steps += 1
    
    print(f"Epoch {epoch+1} | Loss: {total_loss/steps:.4f}")

Precomputing: 100%|██████████| 6497834/6497834 [17:10<00:00, 6308.15it/s] 


: 

**The idea is now that if I take image metadata (such as date), I could better defirentiate the species**

In [ ]:
def predict(image_emb, date_emb, species_text_embs, date_encoder, alpha=0.8):
    """
    image_emb:         (1, 512) from BioCLIP
    date_emb:          (1, 4) cyclical features
    species_text_embs: (N, 512) precomputed for all candidate species
    alpha:             weight for image vs date (tune this)
    """
    with torch.no_grad():
        date_in_clip = date_encoder(date_emb)  # (1, 512)

        image_sim = image_emb @ species_text_embs.T  # (1, N)
        date_sim  = date_in_clip @ species_text_embs.T  # (1, N)

        combined = alpha * image_sim + (1 - alpha) * date_sim
        probs = combined.softmax(dim=-1)

    return probs

# Experiment 2? Same Idea but with location of the image! 